# Text Summarization - 1C
## Artikel: TikTok dan GoTo Resmi Berpisah
**Metode:** TF-IDF dengan Sastrawi (Bahasa Indonesia)

> Artikel: Kemitraan TikTok dan GoTo berakhir pada Juni 2024, saham GOTO turun lebih dari 10%.

## 1. Install & Import Dependencies

In [ ]:
!pip install Sastrawi -q

In [ ]:
import nltk
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import matplotlib.pyplot as plt
import pandas as pd

nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)
print('Dependencies loaded successfully')

## 2. Define Document

In [ ]:
article = """
TikTok dan GoTo Resmi Berpisah, Saham GOTO Anjlok

Kemitraan antara TikTok dan GoTo secara resmi berakhir pada Rabu, 5 Juni 2024. PT GoTo Gojek Tokopedia Tbk mengumumkan bahwa TikTok telah memutuskan untuk tidak melanjutkan perjanjian operasional bersama yang telah berjalan selama kurang lebih setahun terakhir.

Dalam keterbukaan informasi yang diunggah ke Bursa Efek Indonesia (BEI), GoTo menyampaikan bahwa TikTok memilih untuk menjalankan operasional TikTok Shop secara mandiri tanpa melibatkan ekosistem GoTo.

Keputusan ini langsung berdampak pada pergerakan saham GOTO di bursa. Harga saham GOTO anjlok lebih dari 10 persen pada awal perdagangan setelah pengumuman tersebut dirilis ke publik. Investor bereaksi negatif terhadap kabar perpisahan ini karena kemitraan dengan TikTok dianggap sebagai salah satu katalis pertumbuhan utama GoTo.

Sebelumnya, TikTok dan GoTo menjalin kerja sama strategis pada Oktober 2023 setelah pemerintah Indonesia melarang transaksi jual beli di platform media sosial. TikTok Shop saat itu harus tutup, namun kembali beroperasi melalui platform Tokopedia milik GoTo.

Dalam perjanjian tersebut, TikTok menginvestasikan lebih dari 1,5 miliar dolar AS ke GoTo dan mendapatkan kendali mayoritas atas Tokopedia. Kerja sama ini sempat dipandang sebagai solusi win-win: TikTok bisa tetap menjalankan bisnis e-commerce di Indonesia, sementara GoTo mendapat suntikan modal besar.

Namun setelah hampir setahun berjalan, TikTok memutuskan untuk memisahkan operasional TikTok Shop dari ekosistem GoTo. Manajemen GoTo menyatakan pihaknya menghormati keputusan TikTok dan akan terus fokus mengembangkan bisnis inti, termasuk Gojek dan layanan keuangan GoTo Financial.

Analis pasar menilai perpisahan ini cukup mengejutkan, mengingat investasi besar yang sudah dilakukan TikTok. Beberapa analis memperkirakan GoTo perlu mencari mitra strategis baru untuk menggantikan peran TikTok dalam ekosistem e-commerce mereka.

GoTo sendiri menegaskan bahwa kondisi keuangan perusahaan tetap stabil dan mereka memiliki cukup likuiditas untuk menjalankan operasional ke depan. Perusahaan juga menyatakan akan terus berupaya mencapai profitabilitas sesuai target yang telah ditetapkan.
"""

print(f'Article length: {len(article)} characters')

## 3. Text Preprocessing

In [ ]:
# Sentence tokenization
sent_tokens = sent_tokenize(article)

print(f'Total sentences: {len(sent_tokens)}\n')
print('=== List of Sentences ===')
for i, s in enumerate(sent_tokens):
    print(f'{i+1}. {s}')

In [ ]:
# Remove Indonesian stop words using Sastrawi
factory = StopWordRemoverFactory()
stopword_remover = factory.create_stop_word_remover()

cleaned_sentences = [stopword_remover.remove(s) for s in sent_tokens]

print('=== Cleaned Sentences (stop words removed) ===')
for i, s in enumerate(cleaned_sentences):
    print(f'{i+1}. {s}')

## 4. TF-IDF Vectorization

In [ ]:
# Build TF-IDF matrix
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(cleaned_sentences)
feature_names = vectorizer.get_feature_names_out()

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'  → {tfidf_matrix.shape[0]} sentences, {tfidf_matrix.shape[1]} unique words')

In [ ]:
# Show top TF-IDF words for the first sentence
df_first = pd.DataFrame({
    'Word': feature_names,
    'TF-IDF': tfidf_matrix[0].toarray().flatten()
})
df_first = df_first[df_first['TF-IDF'] > 0].sort_values('TF-IDF', ascending=False)

print('=== Top TF-IDF words in Sentence 1 ===')
print(df_first.to_string(index=False))

## 5. Sentence Scoring

In [ ]:
# Calculate average TF-IDF score per sentence
sent_scores = []
for i, row in enumerate(tfidf_matrix):
    total = row.sum()
    n_words = len(row.data)
    avg = total / n_words if n_words > 0 else 0
    sent_scores.append(avg)

print('=== Average TF-IDF Score per Sentence ===')
for i, score in enumerate(sent_scores):
    print(f'Sentence {i+1}: {score:.4f} | {sent_tokens[i][:70]}...')

In [ ]:
# Visualize sentence scores
plt.figure(figsize=(12, 5))
plt.bar(range(1, len(sent_scores)+1), sent_scores, color='seagreen')
plt.xlabel('Sentence Number')
plt.ylabel('Average TF-IDF Score')
plt.title('Average TF-IDF Score per Sentence\n(Artikel: TikTok dan GoTo Resmi Berpisah)')
plt.xticks(range(1, len(sent_scores)+1))
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 6. Text Summarization

In [ ]:
# Threshold = mean score
threshold = sum(sent_scores) / len(sent_scores)
print(f'Threshold (mean score): {threshold:.4f}\n')

summary_sentences = []
print('=== Sentences selected for summary ===')
for i, (sent, score) in enumerate(zip(sent_tokens, sent_scores)):
    if score >= threshold:
        summary_sentences.append(sent)
        print(f'[v] Sentence {i+1} (score={score:.4f}): {sent[:80]}...')

In [ ]:
# Final summary
final_summary = ' '.join(summary_sentences)

print('=== RINGKASAN ARTIKEL ===')
print(final_summary)